# Unsupervised Learning: K-Means Clustering

---

## Introduction

**K-Means** is one of the most widely used unsupervised learning algorithms. It partitions a dataset into `k` clusters by iteratively assigning each sample to its nearest centroid and then updating the centroids to the mean of their assigned samples.

### The Algorithm

| Step | Action |
|---|---|
| 1 | Initialize `k` centroids randomly |
| 2 | Assign each sample to the nearest centroid (Euclidean distance) |
| 3 | Recompute each centroid as the mean of its assigned samples |
| 4 | Repeat steps 2–3 until convergence (centroids stop moving) |

### Choosing k — The Elbow Method

The optimal number of clusters is found using the **Within-Cluster Sum of Squares (WCSS)** — also called inertia. As `k` increases, WCSS always decreases. The **elbow point** — where the rate of decrease sharply slows — is the recommended `k`.

### Workflow

1. Load and visualize the student dataset (2D)
2. Apply the Elbow Method to find optimal `k`
3. Fit K-Means and visualize clusters in 2D
4. Generate a 3D synthetic dataset
5. Apply the Elbow Method in 3D
6. Fit K-Means and visualize clusters in 3D

---

## 1. Importing Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

---

## 2. Dataset — Student CGPA vs. IQ

The student dataset contains two features: `cgpa` (cumulative GPA) and `iq`. We apply K-Means to discover natural groupings of students without using any labels.

In [ ]:
df = pd.read_csv('student_clustering.csv')

print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print('\nMissing values:', df.isnull().sum().sum())
df.describe().round(3)

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df['cgpa'], df['iq'], alpha=0.7, edgecolors='k', linewidths=0.3, s=40)
plt.xlabel('CGPA')
plt.ylabel('IQ')
plt.title('Student Data — CGPA vs. IQ (Unlabelled)')
plt.tight_layout()
plt.show()

---

## 3. Elbow Method — Finding Optimal k (2D)

We compute WCSS (inertia) for `k` from 1 to 10. The elbow point — where adding more clusters yields diminishing returns — indicates the best `k`.

In [ ]:
wcss = []
k_range = range(1, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(df)
    wcss.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(k_range, wcss, 'o-', lw=2, color='steelblue')
plt.axvline(x=4, color='red', linestyle='--', lw=1.5, label='Elbow at k=4')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('WCSS (Inertia)')
plt.title('Elbow Method — Optimal k for Student Dataset')
plt.xticks(k_range)
plt.legend()
plt.tight_layout()
plt.show()

print('WCSS values:')
for k, w in zip(k_range, wcss):
    print(f'  k={k:2d}  WCSS: {w:.2f}')

---

## 4. Fit K-Means with k=4 and Visualize Clusters (2D)

We fit K-Means with the optimal `k=4` found from the elbow plot and visualize the resulting cluster assignments in the CGPA–IQ feature space.

In [ ]:
X = df.values

km = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = km.fit_predict(X)

print(f'Cluster sizes: {pd.Series(labels).value_counts().sort_index().to_dict()}')
print(f'Inertia      : {km.inertia_:.4f}')

In [ ]:
colors = ['steelblue', 'seagreen', 'gold', 'tomato']
cluster_names = ['Cluster 0', 'Cluster 1', 'Cluster 2', 'Cluster 3']

plt.figure(figsize=(8, 5))
for i, (color, name) in enumerate(zip(colors, cluster_names)):
    plt.scatter(
        X[labels == i, 0], X[labels == i, 1],
        color=color, label=name, alpha=0.8,
        edgecolors='k', linewidths=0.3, s=50
    )

# Plot centroids
plt.scatter(
    km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
    color='black', marker='X', s=200, zorder=5, label='Centroids'
)

plt.xlabel('CGPA')
plt.ylabel('IQ')
plt.title('K-Means Clustering (k=4) — Student Data')
plt.legend()
plt.tight_layout()
plt.show()

### 4.1 Cluster Centroids

In [ ]:
centroids_df = pd.DataFrame(
    km.cluster_centers_,
    columns=df.columns,
    index=[f'Cluster {i}' for i in range(4)]
).round(3)

print('Cluster Centroids:')
print(centroids_df)

---

## 5. 3D Clustering — Synthetic Dataset

K-Means extends naturally to higher dimensions. We generate a 3D synthetic dataset using `make_blobs` with 4 well-separated cluster centers to demonstrate K-Means in 3D space with interactive Plotly visualization.

In [ ]:
centroids_3d  = [(-5, -5, 5), (5, 5, -5), (3.5, -2.5, 4), (-2.5, 2.5, -4)]
cluster_std   = [1, 1, 1, 1]

X3d, y3d = make_blobs(
    n_samples=200,
    centers=centroids_3d,
    cluster_std=cluster_std,
    n_features=3,
    random_state=42
)

print('3D dataset shape:', X3d.shape)
print('True cluster sizes:', pd.Series(y3d).value_counts().sort_index().to_dict())

In [ ]:
fig = px.scatter_3d(
    x=X3d[:, 0], y=X3d[:, 1], z=X3d[:, 2],
    title='3D Synthetic Dataset — Before Clustering',
    labels={'x': 'Feature 1', 'y': 'Feature 2', 'z': 'Feature 3'},
    opacity=0.7
)
fig.show()

---

## 6. Elbow Method — Finding Optimal k (3D)

In [ ]:
wcss_3d = []
k_range_3d = range(1, 21)

for k in k_range_3d:
    km3d = KMeans(n_clusters=k, random_state=42, n_init=10)
    km3d.fit(X3d)
    wcss_3d.append(km3d.inertia_)

plt.figure(figsize=(10, 4))
plt.plot(k_range_3d, wcss_3d, 'o-', lw=2, color='steelblue')
plt.axvline(x=4, color='red', linestyle='--', lw=1.5, label='Elbow at k=4')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('WCSS (Inertia)')
plt.title('Elbow Method — Optimal k for 3D Synthetic Dataset')
plt.xticks(k_range_3d)
plt.legend()
plt.tight_layout()
plt.show()

---

## 7. Fit K-Means with k=4 and Visualize Clusters (3D)

In [ ]:
km3d_final = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_3d  = km3d_final.fit_predict(X3d)

print(f'Cluster sizes : {pd.Series(labels_3d).value_counts().sort_index().to_dict()}')
print(f'Inertia       : {km3d_final.inertia_:.4f}')

fig = px.scatter_3d(
    x=X3d[:, 0], y=X3d[:, 1], z=X3d[:, 2],
    color=labels_3d.astype(str),
    title='K-Means Clustering (k=4) — 3D Synthetic Dataset',
    labels={'x': 'Feature 1', 'y': 'Feature 2', 'z': 'Feature 3', 'color': 'Cluster'},
    opacity=0.8
)
fig.show()

---

## Conclusion

This notebook demonstrated K-Means clustering on both a real 2D student dataset and a synthetic 3D dataset, covering the full workflow from data exploration through cluster visualization.

**Key findings:**

- The **Elbow Method** on the student dataset clearly identifies `k=4` as the optimal number of clusters — the WCSS curve bends sharply at this point, after which additional clusters yield diminishing reductions in inertia.
- The **2D cluster plot** shows four well-separated student groups in the CGPA–IQ space. The centroid table quantifies the average profile of each group, which could be used to characterize student archetypes.
- The **3D elbow plot** also identifies `k=4` correctly — matching the four true cluster centers used to generate the data — demonstrating that K-Means scales naturally to higher dimensions.
- The **3D interactive plot** makes the spatial separation of clusters intuitive, which static 2D projections would flatten and obscure.

**Takeaways:**

- K-Means requires specifying `k` in advance. The Elbow Method is a reliable heuristic but should be combined with domain knowledge when available.
- K-Means assumes **spherical, equally sized clusters**. For non-spherical or unequal-density clusters, DBSCAN or Gaussian Mixture Models are better alternatives.
- `n_init=10` (default) runs K-Means with 10 different random initializations and keeps the best — this is important since K-Means is sensitive to initial centroid placement.
- Feature scaling matters for K-Means since it relies on Euclidean distance — features on very different scales will dominate the distance metric. Always apply `StandardScaler` or `MinMaxScaler` before clustering on real datasets with mixed-scale features.